# 02 - yfinance Scraping

This notebook collects market data from yfinance/Yahoo Finance.

Final CSV outputs:
- `data/yfinance_stock_prices.csv`
- `data/yfinance_company_profile.csv`
- `data/yfinance_dividends.csv`

In [6]:
!pip -q install pandas yfinance curl_cffi

In [7]:
from pathlib import Path
import pandas as pd
import yfinance as yf
from curl_cffi import requests as curl_requests
try:
    from IPython.display import display
except ImportError:
    # Local validation fallback. Colab has display(), but plain Python may not.
    def display(value):
        print(value)

DATA_DIR = Path("/content/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

START_DATE = "2021-01-01"
TICKERS = {
    "SRU-UN.TO": "SmartCentres REIT",
    "REI-UN.TO": "RioCan REIT",
    "CHP-UN.TO": "Choice Properties REIT",
    "AP-UN.TO": "Allied Properties REIT",
    "FCR-UN.TO": "First Capital REIT",
}
yf_session = curl_requests.Session(impersonate="chrome")
# Local Windows certificate stores can fail with Yahoo requests. Colab usually
# works normally, but this keeps the notebook robust on this PC too.
yf_session.verify = False

In [8]:
# Daily prices support later return, volatility, and peer performance analysis.
# Adjusted Close is usually the most useful price column for return analysis.
stock_frames = []
for ticker, company_name in TICKERS.items():
    history = yf.Ticker(ticker, session=yf_session).history(start=START_DATE, auto_adjust=False)
    if history.empty:
        print("No stock history returned for", ticker)
        continue
    history = history.reset_index()
    keep_columns = [column for column in ["Date", "Adj Close", "Close", "High", "Low", "Open", "Volume"] if column in history.columns]
    history = history[keep_columns]
    history.insert(0, "company_name", company_name)
    history.insert(0, "ticker", ticker)
    stock_frames.append(history)

stock_prices_df = pd.concat(stock_frames, ignore_index=True) if stock_frames else pd.DataFrame()
stock_prices_df.to_csv(DATA_DIR / "yfinance_stock_prices.csv", index=False)
print("Saved stock prices:", len(stock_prices_df))
display(stock_prices_df.head())

Saved stock prices: 6955


,ticker,company_name,Date,Adj Close,Close,High,Low,Open,Volume
0,SRU-UN.TO,SmartCentres REIT,2021-01-04 00:00:00-05:00,15.667401,23.010000,23.350000,22.760000,23.340000,579100
1,SRU-UN.TO,SmartCentres REIT,2021-01-05 00:00:00-05:00,15.776341,23.170000,23.389999,23.070000,23.080000,426800
2,SRU-UN.TO,SmartCentres REIT,2021-01-06 00:00:00-05:00,15.864856,23.299999,23.420000,23.139999,23.309999,555400
3,SRU-UN.TO,SmartCentres REIT,2021-01-07 00:00:00-05:00,15.871670,23.309999,23.540001,23.240000,23.469999,573800
4,SRU-UN.TO,SmartCentres REIT,2021-01-08 00:00:00-05:00,15.885289,23.330000,23.490000,23.219999,23.360001,314400


In [9]:
# Company profile is a point-in-time peer snapshot.
# Market cap compares size; beta compares market sensitivity; dividend yield compares income.
fields = ["symbol", "shortName", "longName", "marketCap", "beta", "dividendYield", "sharesOutstanding", "trailingPE", "forwardPE", "fiftyTwoWeekHigh", "fiftyTwoWeekLow", "sector", "industry"]
profile_rows = []
for ticker in TICKERS:
    info = yf.Ticker(ticker, session=yf_session).info
    row = {"ticker": ticker}
    for field in fields:
        row[field] = info.get(field)
    profile_rows.append(row)

company_profile_df = pd.DataFrame(profile_rows)
company_profile_df.to_csv(DATA_DIR / "yfinance_company_profile.csv", index=False)
print("Saved profiles:", len(company_profile_df))
display(company_profile_df)

Saved profiles: 5


,ticker,symbol,shortName,longName,marketCap,beta,dividendYield,sharesOutstanding,trailingPE,forwardPE,fiftyTwoWeekHigh,fiftyTwoWeekLow,sector,industry
0,SRU-UN.TO,SRU-UN.TO,SMARTCENTRES REIT,SmartCentres Real Estate Investment Trust,5954195968,0.850,6.10,144708787,14.172896,15.165000,30.90,25.010,Real Estate,REIT - Retail
1,REI-UN.TO,REI-UN.TO,RIOCAN REAL EST UN,RioCan Real Estate Investment Trust,6747986432,0.999,5.00,290616179,27.879519,14.283950,23.25,17.485,Real Estate,REIT - Retail
2,CHP-UN.TO,CHP-UN.TO,CHOICE PROPERTIES REIT,Choice Properties Real Estate Investment Trust,11848783872,0.781,4.76,328024272,NaN,15.590478,16.87,14.090,Real Estate,REIT - Retail
3,AP-UN.TO,AP-UN.TO,ALLIED PROPERTIES REAL ESTATE I,Allied Properties Real Estate Investment Trust,2115181696,1.090,7.07,183955983,NaN,5.598901,22.27,8.740,Real Estate,REIT - Office
4,FCR-UN.TO,FCR-UN.TO,FIRST CAPITAL REIT UNITS,First Capital Real Estate Investment Trust,4873886208,0.885,3.94,212555000,4.549603,16.985186,23.84,18.100,Real Estate,REIT - Retail


In [10]:
# yfinance gives a longer SmartCentres distribution history than the visible company website table.
dividends = yf.Ticker("SRU-UN.TO", session=yf_session).dividends
if dividends.empty:
    yfinance_dividends_df = pd.DataFrame(columns=["ticker", "source", "dividend_date", "dividend_amount", "notes"])
else:
    dividends = dividends[dividends.index >= START_DATE]
    yfinance_dividends_df = dividends.reset_index()
    yfinance_dividends_df.columns = ["dividend_date", "dividend_amount"]
    yfinance_dividends_df.insert(0, "source", "yfinance")
    yfinance_dividends_df.insert(0, "ticker", "SRU-UN.TO")
    yfinance_dividends_df["notes"] = "Longer SmartCentres dividend/distribution history from Yahoo Finance"

yfinance_dividends_df.to_csv(DATA_DIR / "yfinance_dividends.csv", index=False)
print("Saved dividends:", len(yfinance_dividends_df))
display(yfinance_dividends_df.head())

Saved dividends: 66


,ticker,source,dividend_date,dividend_amount,notes
0,SRU-UN.TO,yfinance,2021-01-28 00:00:00-05:00,0.154,Longer SmartCentres dividend/distribution hist...
1,SRU-UN.TO,yfinance,2021-02-25 00:00:00-05:00,0.154,Longer SmartCentres dividend/distribution hist...
2,SRU-UN.TO,yfinance,2021-03-30 00:00:00-04:00,0.154,Longer SmartCentres dividend/distribution hist...
3,SRU-UN.TO,yfinance,2021-04-29 00:00:00-04:00,0.154,Longer SmartCentres dividend/distribution hist...
4,SRU-UN.TO,yfinance,2021-05-28 00:00:00-04:00,0.154,Longer SmartCentres dividend/distribution hist...
